# Vertical Jump Coach Agent — Progress Report
WatSPEED Agentic AI · Course Project · Sean Tang · due Nov 8, 2026  
Code: [github.com/seantanger/thp-jump-coach-agent](https://github.com/seantanger/thp-jump-coach-agent) (branch `main`)

For this project, I am building an AI coach for vertical jump training based on the information from John Evans / THP Strength YouTube video transcripts. The work I have done is with the help of Claude Code. The agents use the Claude API. The first thing I did was collect all of the transcripts from YouTube using yt-dlp, and preprocess the text with regex rules. I then tried three chunking techniques: fixed-size character splitting (LangChain's `RecursiveCharacterTextSplitter`) at two chunk sizes, and a third method that cuts the transcript into ~60-second windows at natural pauses in speech. After embedding the chunks using nomic embed in Ollama, I made a simple LangGraph loop where the Coach calls a search tool itself (up to 4 searches, rewriting its query if needed) and answers with the video and timestamp. I compared each of the three chunking methods with three search methods (keyword/BM25, vector, and a hybrid of the two) on 30 test questions. The best combination was the larger character chunks with vector search. I also built a second agent, a Planner, that writes multi-week training programs from reviewed templates, runs them through code checks, and waits for my approval before saving (this is still a WIP).

I went through a few versions of the Coach prompt, most recently to make answers more concise. For safeguards, a code check rejects any answer that cites a source the search didn't return, a third agent (the Verifier) checks each point against its source in a fresh context, and a code safety check sends messages with red-flag pain symptoms (like a pop in the knee) to a physio instead of coaching them.

Next Steps/Ideas:
- Plan quality: reading the first plan, sessions needed more detail. Month 1 accessories are now named exercises (the template lists options such as RDL, hip thrust and Nordic curls). Still to do: the same for the general-circuit days and months 2–6, a load for every lift, and warm-up and ramp-up sets instead of one weight for every set. I also need to still work on the re-editing plan process. For example, allow the user to keep reiterating the plan if they have suggestions.
- Connect the Planner and Coach. Right now they are two separate LangGraph graphs: a code router detects plan requests, but the chat doesn't pass them to the Planner yet. I want to maybe combine them into one parent graph with the Coach and Planner as subgraphs.
- Give the Coach access to the current plan and profile, so its answers fit where the athlete is in the program.
- Memory: right now the profile and each plan version are saved as JSON files. Would something like Letta be better?
- Training log: record sessions as done, changed or missed, plus pain and jump results, and use them to decide when the athlete moves to the next stage. (Not sure if this should be like a UI or the user should enter a chat to tell the system that they have finished a workout)
- Workout variety: the Planner can now swap exercises (at most 2 per week, each cited and equipment-checked); next is more options per slot and loosening sets and reps.
- Optional: MCP server.


## Demos
These call the Claude API (about 4¢ for the Coach question, about 25¢ for a plan) and read the transcript index. The outputs are saved in this notebook.

In [1]:
import os, sys
from pathlib import Path

ROOT = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()) / "jump-coach-agent"
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
os.environ.setdefault("RETRIEVAL_MODE", "dense")  # dense (vector) search needs Ollama running

'dense'

**The Coach's LangGraph setup** ([`src/coach/graph.py`](https://github.com/seantanger/thp-jump-coach-agent/blob/main/jump-coach-agent/src/coach/graph.py)). Two nodes: `coach` (the LLM with tools bound) and `tools` (runs the searches, capped at 4). The graph loops coach → tools → coach until the LLM stops calling tools. `answer_question` (used below) builds this graph, runs it, checks the citations, and sends the answer to the Verifier.

**1. Ask the Coach a question.** The Coach searches the library, answers with citations, then the Verifier checks it. (The two agents are linked through code, not the LangGraph)

In [2]:
from src.coach.qa import answer_question

r = answer_question("Should I static stretch before jumping?")
print(r["compact"])
print("\nsearches:", r["queries"])
print("citation check passed:", r["check"]["passed"], "| Verifier:", r["check"].get("verifier", {}).get("verdict"))

Generally no. Skip static stretching right before jumping and use dynamic flexibility instead. If you need it for mobility, do short holds and follow them with something dynamic or ballistic, or do them after the session.

- Static stretching before jumping can temporarily lower force and power output, so keep the pre-jump warm-up dynamic (RDLs, side squats, leg swings, long lunges). [1][2]
- The power drop is dose-dependent. Long, intense stretching hurts most, and 30–60 seconds followed by something ballistic largely cancels the negative effect. [3][4]
- If tight ankles or hips limit your squat or jump positions, do static stretches after your warm-up, or in the evening. [5][4]

**Sources**
1. [John Evans – A Practical 2 Day Per Week Jump System Guaranteed To Get You Dunking @ 01:42](https://www.youtube.com/watch?v=1sQ-FOxDP5M&t=102s)
2. [THP Strength – World Record Vertical Jumper's Exact Warm-Up Routine (50.5 Inches) @ 00:51](https://www.youtube.com/watch?v=O4piVLyfvSA&t=51s)
3. [J

A pain question: the answer sticks to what the library says about tendon pain, says what it doesn't cover, and ends with the physio line. Note that regular AI models typically recommend ice and anti inflammatories which are NOT recommended by THP.

In [3]:
r = answer_question("What should I do if I have knee pain?")
print(r["compact"])
print("\nsearches:", r["queries"])
print("citation check passed:", r["check"]["passed"], "| Verifier:", r["check"].get("verifier", {}).get("verdict"))


If your knee pain is activity-related tendon ache under the kneecap, take 24–48 hours off, then track your morning pain. If it stays above 3/10, start daily isometrics.

- For an early or reactive flare-up, taking 24 to 48 hours off often settles the tendon. You can then return to normal activity without new exercises. [1]
- Test pain each morning with a deep bodyweight squat, scored 1–10. Pain peaks 24–48 hours after a jump session. If the score is above 3/10, start isometrics. [2][3]
- Isometrics: hold a quad exercise such as a leg extension for 45 seconds at about 70% effort. Do 3–5 sets, about three times a day, spaced roughly 6 hours apart. [4][5]

*See a physio if pain is sharp/stabbing or steadily worsening, occurs at rest or wakes you at night, lasts over two weeks without improving despite cutting load, or follows a fall/impact; or if you have swelling, bruising, deformity, a pop, giving way, locking, catching, numbness, tingling, radiating weakness or limping. Back pain with 

**2. Safety check (code, no LLM).** Red-flag symptoms get a physio referral before any agent runs; plan requests are flagged for the Planner (not yet wired into the chat).

In [4]:
from src.red_flags import referral_kind
from src.router import route

In [5]:
for m in ["Should I static stretch before jumping?", "I felt a pop in my knee this week",
          "Make me a plan for the next 12 weeks"]:
    print(f"{m:42} referral={referral_kind(m)}  route={route(m)}")

Should I static stretch before jumping?    referral=None  route=coach
I felt a pop in my knee this week          referral=physio  route=coach
Make me a plan for the next 12 weeks       referral=None  route=program


**3. First plan from the Planner.** Needs a profile first (`uv run python -m scripts.plan profile`). The Planner drafts a plan, the code checks and Verifier run, then the graph pauses for my approval.

**The Planner's LangGraph setup** (`build_plan_graph` in [`src/planner/graph.py`](https://github.com/seantanger/thp-jump-coach-agent/blob/main/jump-coach-agent/src/planner/graph.py)). `context` loads the profile and current plan, `planner` ⇄ `tools` is the LLM tool loop, `review` runs the 8 code checks and the Verifier (failures go back to the planner, at most 2 retries), `approve` pauses the graph until I approve, reject or revise, and `save` writes a new plan version.

**The athlete profile (me).** This is what the Planner reads with its `get_profile` tool. It's stored as `profile.json` and made with `uv run python -m scripts.plan profile`.

In [6]:
from src.planner.store import load_profile

print(load_profile().model_dump_json(indent=2))

{
  "training_age_years": 4.0,
  "lifts": {
    "back_squat_lb": 225.0,
    "half_squat_lb": 245.0,
    "power_clean_lb": 175.0
  },
  "standing_vertical_in": null,
  "approach_vertical_in": 32.0,
  "standing_reach_in": 86.5,
  "height_in": 67.0,
  "days_per_week": 7,
  "available_days": [
    "mon",
    "tue",
    "wed",
    "thu",
    "fri",
    "sat",
    "sun"
  ],
  "session_minutes": 90,
  "equipment": [
    "barbell",
    "rack",
    "platform",
    "box",
    "med ball",
    "safety bar",
    "belt squat",
    "bench",
    "hamstring curl machine"
  ],
  "event": null,
  "goal": "Dunk off two feet. Approach vertical about 31-33 in, standing reach about 7 ft 2-3 in (needs roughly 10 ft 6 in touch). Back squat is a deep squat; half squat estimated.",
  "knee_pain": {
    "status": "ache",
    "notes": "Knee pain in the past, now manageable."
  },
  "injury_notes": "Two-foot jumper. No sport or other jump load during the week."
}


In [7]:
from src.planner import run
from src.planner.render import render_proposal

plan = run.start("new")
print(render_proposal(plan))

What changes:
  New plan outline
  Month 1: volume
  Month 2: intensification (power and 6–8 rep strength)
  Month 3: absolute strength
  (+5 more)
Why: New 6-month long-conjugate outline of six mesocycles, running 9 Nov 2026 to 25 Apr 2027. Weeks 1–2 of Month 1 are written day by day (9–22 Nov). Each week has lifts Mon/Wed/Fri, general days Tue/Thu and a jump session on Sat.
⚠ weeks[2026-11-09]: partly supported — Main lifts, jump 35+5 min and isometrics are supported. Accessory 3x10 and circuit details are not stated in the cited chunks.


In [8]:
plan = run.resume(plan.thread_id, "approve")  # or "reject", or "revise" with comment="..."
print(render_proposal(plan))

Saved as version 1.


**The saved plan in detail.** The 6-month outline and every week written so far (the Planner writes 2 weeks at a time and rolls forward), then the videos the plan draws on. Every session cites the transcript chunks it came from; those are hidden here and shown in the Developer view. (Note that this is still a WIP)

In [9]:
from src.kb.evidence import chunk_meta
from src.planner.render import render_outline, render_week
from src.planner.store import load_program
from src.planner.templates import TEMPLATE_METHOD

program = load_program()
print(f"Plan version {program.version}: {program.change_summary}\n")
print(render_outline(program) + "\n")
for week in program.weeks:
    print(render_week(program, week.start) + "\n")

cites = {c for m in program.mesocycles for c in m.cites} | {
    c for w in program.weeks for d in w.days for s in d.sessions for c in s.cites}
videos = {}  # one line per video the plan draws on
for meta in chunk_meta(cites, method=TEMPLATE_METHOD).values():
    videos.setdefault(meta["title"], meta["url_t"])
print("Videos used:")
for title, url in sorted(videos.items()):
    print(f"  {title}  {url}")


Plan version 1: New 6-month long-conjugate outline of six mesocycles, running 9 Nov 2026 to 25 Apr 2027. Weeks 1–2 of Month 1 are written day by day (9–22 Nov). Each week has lifts Mon/Wed/Fri, general days Tue/Thu and a jump session on Sat.

Month 1 · Mon Nov 9 – Sun Dec 6 · volume · Build work capacity with general work that looks little like jumping; unload in week 4.
Month 2 · Mon Dec 7 – Sun Jan 3 · intensification (power and 6–8 rep strength) · Intensify the month-1 volume with clean pulls and power cleans; heavier RDL and calf raise accessories prime month 3.
Month 3 · Mon Jan 4 – Sun Jan 31 · absolute strength · Heavy squats and heavy clean pulls; eccentric accessories prime month 4.
Month 4 · Mon Feb 1 – Sun Feb 28 · strength-speed (fast bars, eccentric overload) · Power clean to maxes; accentuated eccentric squats and eccentric split squats; general days get more general.
Month 5 · Mon Mar 1 – Sun Mar 28 · speed-strength (bar speed) · Move bars fast (lighter cleans, power sna